# Dunnhumby seed 43: 고CLV 조건부 N M2·M4 빠른 방향성 스크린

기존 **K=1 seed 43 M1** 결과를 입력·분할·학습계약 검증 후 재사용하고, 새로 학습하는 것은 M2-HN과 M4-HN 두 arm뿐입니다. 신규상품 과업, `MIN_ITEM_INTER=1`, binary graph, 균등 음성 1개 BPR, 고정 100 epoch를 사용합니다.

이 실행은 단일 개발 seed의 방향성 확인입니다. 성공·실패, CLV 귀속, 안정성, 유의성 또는 일반화를 확정하지 않습니다. 런타임이 끊기면 Drive를 다시 마운트하고 **학습 셀을 다시 실행**하세요. 마지막 완료 epoch 다음부터 자동 재개합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'fe7290e'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA

import subprocess
current = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
assert current.startswith(REVIEWED_SHA), (current, REVIEWED_SHA)

In [ ]:
import importlib
import json
import torch
import lightgcn_clv_high_clv_n_quick_screen as screen
screen = importlib.reload(screen)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert screen.CODE_VERSION == 'high-clv-candidate-n-m2-m4-quick-screen-v1'
cfg = screen.configure_quick_screen()
print(json.dumps(screen.preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
# 런타임 중단 후에는 이 셀을 다시 실행하면 epoch 단위로 자동 재개됩니다.
screen = importlib.reload(screen)
cfg = screen.configure_quick_screen()
result_df = screen.run_quick_screen(cfg)

In [ ]:
import json
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 300)

print('1) M1·M2-HN·M4-HN 전체 절대지표')
display(result_df)
print('2) M1 대비 전체 지표 비교')
comparison = pd.DataFrame(result_df.attrs['comparison'])
display(comparison)
print('3) 단일시드 방향성 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('4) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))